# Aula 6 — Um Sistema de Ponta a Ponta

**O produto desta aula é a pasta `sistema/`, que você roda no VS Code.**
Este notebook é o laboratório: aqui você testa os dois modelos sozinhos,
sem tela no caminho, antes de montá-los no painel.

Ele também é o plano B: se o seu computador der trabalho na instalação,
acompanhe a parte de modelagem por aqui e monte o painel depois, com
calma.

A **Parte A** é a demonstração. A **Parte B** é com você, nos lugares
marcados com `# SEU CODIGO AQUI`.

## Parte A: Demonstração

In [ ]:
!pip install prophet -q

### Os dados do Clube do Café

Duas tabelas do mesmo negócio: os assinantes e a receita diária.

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.linear_model import LogisticRegression

BASE = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/"
# Alternativa para testar offline:
# BASE = "../../data/"

clientes = pd.read_csv(BASE + "clube_cafe_clientes.csv")
vendas = pd.read_csv(BASE + "clube_cafe_vendas.csv", parse_dates=["data"])

print(f"{len(clientes)} assinantes, {clientes['cancelou'].mean():.1%} cancelaram")
print(f"{len(vendas)} dias de receita, média de R$ {vendas['receita'].mean():.2f} por dia")
clientes.head()

### Modelo 1: quem corre risco de cancelar

É a regressão logística da Aula 3, com as colunas deste negócio:

$$P(\text{cancelar}) = \sigma(z) \qquad z = w_0 + w_1 x_1 + \dots + w_p x_p$$

As funções abaixo são exatamente as que vão para o `modelos.py`.

In [ ]:
COLUNAS_DO_CLIENTE = ["meses_de_casa", "valor_mensal", "entregas_atrasadas", "plano"]

def preparar_tabela(clientes):
    # O plano é um nome, não um número: vira colunas de 0 e 1
    tabela = pd.get_dummies(clientes[COLUNAS_DO_CLIENTE], columns=["plano"],
                            drop_first=True)
    return tabela.astype(float)

def treinar_classificador(clientes):
    tabela = preparar_tabela(clientes)
    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(tabela, clientes["cancelou"])
    return modelo

def calcular_risco(modelo, clientes):
    tabela = preparar_tabela(clientes)
    return modelo.predict_proba(tabela)[:, 1]

classificador = treinar_classificador(clientes)

for nome, peso in zip(preparar_tabela(clientes).columns, classificador.coef_[0]):
    print(f"{nome}: peso {peso:.3f}, multiplica a chance por {np.exp(peso):.2f}")

### Da probabilidade para a lista de ligações

O limiar transforma probabilidade em decisão. No painel ele vira um
controle deslizante, porque quem escolhe o tamanho da operação é quem vai
fazer as ligações.

In [ ]:
risco = calcular_risco(classificador, clientes)

for limiar in [0.4, 0.5, 0.6]:
    na_lista = risco >= limiar
    receita_em_risco = clientes.loc[na_lista, "valor_mensal"].sum()
    acerto = clientes.loc[na_lista, "cancelou"].mean()
    print(f"limiar {limiar}: {na_lista.sum():3d} clientes | "
          f"R$ {receita_em_risco:8.2f} de receita mensal | "
          f"{acerto:.0%} da lista cancelou de fato")

### Modelo 2: quanto o clube vai faturar

É o Prophet da Aula 5, com a lista de feriados:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

In [ ]:
def treinar_previsor(vendas):
    # O Prophet exige as colunas com os nomes ds e y
    serie = vendas.rename(columns={"data": "ds", "receita": "y"})
    feriados = pd.DataFrame({
        "holiday": "feriado",
        "ds": vendas[vendas["feriado"] == 1]["data"],
    })
    modelo = Prophet(holidays=feriados)
    modelo.fit(serie[["ds", "y"]])
    return modelo

def prever(modelo, dias):
    futuro = modelo.make_future_dataframe(periods=dias)
    return modelo.predict(futuro)

previsor = treinar_previsor(vendas)
previsao = prever(previsor, 30)
proximos_30 = previsao.tail(30)

print(f"Receita prevista em 30 dias: R$ {proximos_30['yhat'].sum():.2f}")
print(f"Média por dia:               R$ {proximos_30['yhat'].mean():.2f}")
print(f"Pior caso da faixa:          R$ {proximos_30['yhat_lower'].sum():.2f}")

In [ ]:
figura = previsor.plot(previsao)

### De notebook para sistema

As seis funções acima são o `modelos.py` inteiro. O `app.py` só chama
essas funções e desenha o resultado na tela:

| No notebook | No painel |
|---|---|
| `calcular_risco` | a tabela da primeira aba |
| o laço dos limiares | o controle deslizante |
| `prever` | o gráfico da segunda aba |

É essa a passagem que a aula faz: o que você testou aqui vira uma tela
que outra pessoa consegue usar.

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo os assinantes

Rode a célula e observe: qual plano cancela mais? Isso bate com a sua
intuição?

In [ ]:
resumo = clientes.groupby("plano")[["valor_mensal", "entregas_atrasadas", "cancelou"]].mean()
print(resumo.round(2))

In [ ]:
if len(clientes) == 400:
    print(f"✅ A base tem {len(clientes)} assinantes, como esperado.")
else:
    print("❌ Confira se você rodou a célula que carrega os dados.")

### Exercício 2: o dia mais fraco da semana

Rode e observe. O clube entrega de segunda a sexta: dá para ver isso nos
dados?

In [ ]:
vendas["dia_da_semana"] = vendas["data"].dt.weekday
nomes = ["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"]
media_por_dia = vendas.groupby("dia_da_semana")["receita"].mean()
for numero, media in media_por_dia.items():
    print(f"{nomes[numero]}: R$ {media:.2f}")

In [ ]:
print("Converse com um colega: por que o fim de semana fatura menos neste negócio?")

### Exercício 3: o seu classificador

Treine um classificador chamado `meu_classificador` usando a função
`treinar_classificador` já pronta.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print("Colunas usadas:", list(meu_classificador.feature_names_in_))

In [ ]:
if hasattr(meu_classificador, "coef_") and len(meu_classificador.coef_[0]) == 5:
    print("✅ Classificador treinado com as cinco colunas.")
else:
    print("❌ Confira se você passou a tabela de clientes inteira para a função.")

### Exercício 4: a lista de risco com limiar 0,40

Calcule o risco de todo mundo, monte a lista com limiar de 0,40 e some a
receita mensal dessa lista.

In [ ]:
LIMIAR = 0.40

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Clientes na lista: {len(minha_lista)}")
print(f"Receita mensal em risco: R$ {receita_em_risco:.2f}")

In [ ]:
if 150 < len(minha_lista) < 190:
    print("✅ A lista ficou do tamanho esperado para esse limiar.")
else:
    print("❌ Confira o sinal da comparação: o risco precisa ser maior ou igual ao limiar.")

### Exercício 5: prevendo 60 dias

Use a função `prever` para olhar 60 dias à frente e some a receita
prevista desse período.

In [ ]:
DIAS = 60

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Receita prevista para {DIAS} dias: R$ {total_previsto:.2f}")
print(f"Pior caso da faixa: R$ {proximos['yhat_lower'].sum():.2f}")

In [ ]:
if 150000 < total_previsto < 220000:
    print("✅ O total ficou na faixa esperada para 60 dias.")
else:
    print("❌ Confira se você pegou só as últimas 60 linhas da previsão.")

### Exercício 6: o risco de um cliente novo

Esta é a função que o simulador do painel usa. Complete a linha do
`reindex`, que põe as colunas na mesma ordem do treino.

In [ ]:
cliente_novo = pd.DataFrame({
    "meses_de_casa": [4],
    "valor_mensal": [129.0],
    "entregas_atrasadas": [3],
    "plano": ["Premium"],
})
tabela_do_cliente = pd.get_dummies(cliente_novo, columns=["plano"])

In [ ]:
# SEU CODIGO AQUI

In [ ]:
risco_do_novo = meu_classificador.predict_proba(tabela_do_cliente.astype(float))[0][1]
print(f"Risco de cancelar: {risco_do_novo:.0%}")

In [ ]:
if 0.4 < risco_do_novo < 0.99:
    print("✅ Risco alto, como esperado para quem tem pouco tempo de casa e três atrasos.")
else:
    print("❌ Sem o reindex, o modelo recebe as colunas trocadas e devolve um número sem sentido.")

### Exercício 7: desafio, qual limiar você recomenda?

Cada ligação de retenção custa R\\$ 15. Um assinante perdido custa doze
mensalidades. Rode o laço abaixo e escolha o limiar que você levaria para
a reunião.

In [ ]:
for limiar in [0.3, 0.4, 0.5, 0.6, 0.7]:
    na_lista = meu_risco >= limiar
    custo_das_ligacoes = na_lista.sum() * 15
    perdidos_evitados = clientes.loc[na_lista, "cancelou"].sum()
    valor_salvo = (clientes.loc[na_lista & (clientes["cancelou"] == 1), "valor_mensal"].sum() * 12)
    print(f"limiar {limiar}: {na_lista.sum():3d} ligações (R$ {custo_das_ligacoes:5d}) | "
          f"{perdidos_evitados:3d} de quem ia cancelar | "
          f"até R$ {valor_salvo:9.2f} salvos")

In [ ]:
print("Não existe resposta única: o limiar certo depende de quanto vale cada cliente.")

Agora, em texto: escolha um limiar e escreva 2 a 3 frases justificando
para a dona do Clube do Café, com os números do laço acima. Depois abra o
`app.py` e mude o valor padrão do controle para o limiar que você
escolheu. Edite esta célula (duplo clique nela) e escreva sua resposta no
lugar deste parágrafo.